# 英文命名实体识别 · C 的 BERT 实验

在 CoNLL-2003 英文数据上独立微调两次 `bert-base-cased`。**E2** 使用全部 14,041 个训练句，**E4** 使用 A、B 共用清单中的 1,404 个训练句。两次都按完整验证集的实体级 micro-F1 选择模型。最后为 A 导出逐句词级预测，为 B 提供可重现的实体抽取演示。

### 开始前

1. 在 [GCP Colab Enterprise](https://console.cloud.google.com/agent-platform/colab/) 选择课程项目和 T4 所在区域，在 **My notebooks → Import → Your computer** 选择本文件。打开后，在 **Connect** 旁选择 **Connect to a runtime**，使用课程提供的 T4 运行时或模板。
2. **现在就能做**：从头顺序执行。数据单元会获取计划指定的三个公开文件。没有 A 的清单时，E2 仍可训练、预测和导出，E4 会显示等待提示。
3. **拿到清单后**：用左侧文件面板上传 A 的 `train_10pct_ids.json` 至配置指定的位置。在同一内核中重新执行 **E4 训练单元**以及它下面的导出、演示和打包单元。内核重启后可从头执行，配置匹配的已完成训练可复用。
4. 最后从文件面板下载交接包与模型包。运行时删除会清除磁盘文件；重要产物请及时保存到本机或课程的 Cloud Storage。

[Google Cloud 导入指南](https://docs.cloud.google.com/colab/docs/create-console-quickstart#import_a_notebook) · [运行时和文件存储](https://docs.cloud.google.com/colab/docs/runtimes) · [Hugging Face 词分类指南](https://huggingface.co/docs/transformers/tasks/token_classification)

## 1 · 准备 GPU 运行环境

在老师提供的 T4 运行时执行以下单元格。安装单元沿用已有的 GPU 版 PyTorch，补齐文本分类、Parquet 和评分依赖。如果界面在首次安装后提示重启内核，请重启并从头运行。下载需要运行时能访问 PyPI 和 Hugging Face。

In [ ]:
# 在 Colab Enterprise 的 IPython 内核运行。已有 PyTorch 时沿用老师配置的版本。
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "torch==2.7.1",
        "--index-url", "https://download.pytorch.org/whl/cu126",
    ])

%pip install -q "transformers==4.57.6" "pyarrow>=16"
%pip install -q --use-pep517 "seqeval==1.2.2"

In [ ]:
import hashlib
import json
import os
import platform
import tempfile
from importlib.metadata import version
from pathlib import Path

import torch
from packaging.version import Version

if Version(torch.__version__.split("+")[0]) < Version("2.4"):
    raise RuntimeError(
        f"当前 PyTorch {torch.__version__}。训练单元使用 torch.amp.GradScaler，"
        "请切换到预装 GPU 版 PyTorch 2.4 或更新版本的运行时，或更新后重启内核。"
    )
if not torch.cuda.is_available():
    raise RuntimeError(
        "当前内核没有可用的 CUDA GPU。请检查已连接的运行时是否使用老师的 T4 模板，"
        "并确认其中安装了 GPU 版 PyTorch，然后从头运行。"
    )

print(f"GPU  ·  {torch.cuda.get_device_name(0)}")
print(f"显存 ·  {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print(f"Python {platform.python_version()}  |  torch {torch.__version__}  |  CUDA {torch.version.cuda}")
print("  |  ".join(f"{p} {version(p)}" for p in ("transformers", "pyarrow", "seqeval")))

## 2 · 实验设置

默认将数据与训练结果保存在当前运行时的 `ner_c_work/`。如有已挂载的持久化磁盘，可将 `WORK_DIR` 改为磁盘中的目录。需要手动提供数据时，可把三份原始 Parquet 放进 `RAW_DIR`，保留原文件名。如需离线加载模型，可把 `MODEL_ID` 改为本地完整模型目录，其中应包含该固定版本的模型权重、配置和快速分词器文件。默认用同一初始权重、随机种子和超参数分别训练 E2 与 E4。

A 的共享清单应包含恰好 1,404 个训练句子的原始 ID。可上传到 `SHARED_IDS` 显示的位置，也可将该变量改为真实上传路径。缺少清单时可以先跑 E2，收到清单后再跑 E4。

In [ ]:
WORK_DIR = Path("ner_c_work").expanduser().resolve()
RAW_DIR = WORK_DIR / "data" / "raw"
RUNS_DIR = WORK_DIR / "runs"
SHARED_IDS = WORK_DIR / "data" / "train_10pct_ids.json"

MODEL_ID = "google-bert/bert-base-cased"
MODEL_REVISION = "cd5ef92a9fb2f889e972770a36d4ed042daf221e"
DATA_REVISION = "e8237f8875e4ecf7f60b07ae49691b85cf893410"
SEED = 42
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 3e-5
MAX_LENGTH = 128

RAW_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("工作目录", WORK_DIR)
print("A 的共享清单", SHARED_IDS)

## 3 · 读取固定版本的 CoNLL-2003

三个划分使用数据仓库同一提交。本单元核对文件摘要、句数、标签名称和金标的 IOB2 边界。每条记录保留原始的字符串 ID，ID 在各自的划分内唯一。相同数值的训练 ID 和测试 ID 分别对应不同句子。[固定版本的 Parquet 文件](https://huggingface.co/datasets/eriktks/conll2003/tree/e8237f8875e4ecf7f60b07ae49691b85cf893410/conll2003) · [标签与划分说明](https://huggingface.co/datasets/eriktks/conll2003)

In [ ]:
import shutil
from urllib.request import urlopen
import pyarrow.parquet as pq

LABELS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
id2label = dict(enumerate(LABELS))
label2id = {name: i for i, name in id2label.items()}

DATA_FILES = {
    "train": (14041, "1c14a0d34accb2c0b1ec61b732000a323019eb63fdd3ca86bddc2b05a2ec0d04"),
    "validation": (3250, "48d75b38d9b08a850e0b31d61d69dd6816687ed6017f347a22afcecaf673ad15"),
    "test": (3453, "47156dc0fe55dd2965fbf92e125b1d04d867560fbaae6d7e2576bd5f021c623e"),
}

def read_split(name):
    count, digest = DATA_FILES[name]
    path = RAW_DIR / f"conll2003-{name}.parquet"
    url = f"https://huggingface.co/datasets/eriktks/conll2003/resolve/{DATA_REVISION}/conll2003/{path.name}"
    if not path.exists():
        with tempfile.NamedTemporaryFile(dir=RAW_DIR, delete=False, suffix=".part") as fp:
            tmp = Path(fp.name)
        try:
            with urlopen(url, timeout=60) as response, tmp.open("wb") as out:
                shutil.copyfileobj(response, out)
            if hashlib.sha256(tmp.read_bytes()).hexdigest() != digest:
                raise ValueError(f"{name} 下载文件与固定版本的摘要不符，请检查下载或手动上传 {url}")
            tmp.replace(path)
        finally:
            tmp.unlink(missing_ok=True)
    if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        raise ValueError(f"请用固定版本文件替换 {path}，下载地址 {url}")
    table = pq.ParquetFile(path).read(columns=["id", "tokens", "ner_tags"])
    features = json.loads(table.schema.metadata[b"huggingface"])["info"]["features"]
    if features["ner_tags"]["feature"]["names"] != LABELS:
        raise ValueError(f"{name} 的标签顺序与模型设置不一致")
    rows = table.to_pylist()
    if len(rows) != count or len({r["id"] for r in rows}) != count:
        raise ValueError(f"{name} 的句子数量或 ID 唯一性异常")
    for row in rows:
        tags = row["ner_tags"]
        if not row["tokens"] or len(row["tokens"]) != len(tags):
            raise ValueError(f"{name}#{row['id']} 原词与标注长度异常")
        previous = "O"
        for tag_id in tags:
            tag = id2label[tag_id]
            if tag.startswith("I-") and previous not in ("B-" + tag[2:], tag):
                raise ValueError(f"{name}#{row['id']} 金标不符合 IOB2")
            previous = tag
    return rows, {"count": count, "sha256": digest, "url": url, "path": str(path)}

splits, data_provenance = {}, {}
for split_name in DATA_FILES:
    splits[split_name], data_provenance[split_name] = read_split(split_name)
    print(f"{split_name:10} {len(splits[split_name]):,} 句")

### 小组共享的 10% 样本

支持 JSON 数组（如 `["8", "25", ...]`）和带 `ids` 或 `train_ids` 键的对象。逐项校验 ID 是否来自训练集。摘要按训练集原始顺序计算，因此同一批 ID 的排列顺序不会改变摘要。A 与 B 应使用同一份清单。

In [ ]:
def load_shared_subset():
    path = Path(SHARED_IDS).expanduser()
    if not path.is_file():
        print(f"尚未收到 A 的 10% 清单。可以先运行 E2。收到后将清单放在 {path} 并运行 E4。")
        return None, {"source": "missing", "path": str(path), "sha256": None, "count": 0}
    contents = path.read_bytes()
    parsed = json.loads(contents.decode("utf-8-sig"))
    if isinstance(parsed, dict):
        parsed = parsed.get("ids", parsed.get("train_ids"))
    if not isinstance(parsed, list) or not all(type(x) in (str, int) for x in parsed):
        raise ValueError("共享清单应为字符串或整数 ID 的数组，或含 ids / train_ids 数组的对象")
    selected = {str(x) for x in parsed}
    if len(parsed) != 1404 or len(selected) != len(parsed):
        raise ValueError("共享清单须包含恰好 1,404 个互不重复的训练 ID")
    rows = [row for row in splits["train"] if row["id"] in selected]
    if len(rows) != len(selected):
        unknown = sorted(selected.difference(row["id"] for row in rows))
        raise ValueError(f"共享清单包含训练集之外的 ID，例如 {unknown[:5]}")
    canonical_ids = json.dumps([row["id"] for row in rows], separators=(",", ":")).encode()
    provenance = {"source": "shared_ids", "path": str(path), "count": len(rows),
                  "sha256": hashlib.sha256(canonical_ids).hexdigest(),
                  "file_sha256": hashlib.sha256(contents).hexdigest()}
    print(f"E4 已匹配 {len(rows):,} 个训练句子，ID 摘要 {provenance['sha256']}")
    return rows, provenance

## 统一实体评分

按句解码 BIO 标签。句首或类型不连续的 `I-X` 视为 `B-X`。只有类型、起点和右开终点均相同才算命中。所有分数以 0～1 表示；表中乘以 100 的差值单位为 F1 点。

In [ ]:
from collections import Counter

ENTITY_TYPES = ("PER", "ORG", "LOC", "MISC")

def decode_spans(tags):
    """Return (start_word, end_word_exclusive, type) within one sentence."""
    spans = []
    start, active = 0, None
    for i, tag in enumerate([*tags, "O"]):
        if tag == "O":
            prefix, kind = "O", None
        else:
            prefix, sep, kind = tag.partition("-")
            if not sep or prefix not in ("B", "I") or kind not in ENTITY_TYPES:
                raise ValueError(f"无效 BIO 标签 {tag!r}，词索引 {i}")
        if prefix == "I" and active == kind:
            continue
        if active is not None:
            spans.append((start, i, active))
        start, active = i, kind
    return spans

def canonical_bio(tags):
    repaired = ["O"] * len(tags)
    for start, end, kind in decode_spans(tags):
        repaired[start] = "B-" + kind
        repaired[start + 1:end] = ["I-" + kind] * (end - start - 1)
    return repaired

def entity_metrics(hits, n_predicted, n_gold):
    return {
        "tp": hits, "fp": n_predicted - hits, "fn": n_gold - hits, "support": n_gold,
        "precision": hits / n_predicted if n_predicted else 0.0,
        "recall": hits / n_gold if n_gold else 0.0,
        "micro_f1": 2 * hits / (n_predicted + n_gold) if n_predicted + n_gold else 0.0,
    }

def score_predictions(rows, pred_sequences):
    if len(rows) != len(pred_sequences):
        raise ValueError("预测句数与数据不符")
    predicted, gold, hits = Counter(), Counter(), Counter()
    length_support, length_hits = Counter(), Counter()
    for row, tags in zip(rows, pred_sequences):
        if len(row["tokens"]) != len(tags) or len(row["ner_tags"]) != len(tags):
            raise ValueError(f"句子 {row['id']} 的词和标签数量不匹配")
        actual = set(decode_spans([LABELS[i] for i in row["ner_tags"]]))
        guess = set(decode_spans(tags))
        matched = actual & guess
        gold.update(s[2] for s in actual)
        predicted.update(s[2] for s in guess)
        hits.update(s[2] for s in matched)
        length_support.update(min(3, e - b) for b, e, _ in actual)
        length_hits.update(min(3, e - b) for b, e, _ in matched)
    scores = entity_metrics(sum(hits.values()), sum(predicted.values()), sum(gold.values()))
    scores["per_type"] = {t: entity_metrics(hits[t], predicted[t], gold[t]) for t in ENTITY_TYPES}
    scores["by_length"] = {
        name: {"support": length_support[n], "tp": length_hits[n],
               "recall": length_hits[n] / length_support[n] if length_support[n] else 0.0}
        for n, name in ((1, "1"), (2, "2"), (3, "3+"))
    }
    return scores

assert decode_spans(["I-ORG", "I-ORG", "B-ORG", "I-LOC", "O"]) == [
    (0, 2, "ORG"), (2, 3, "ORG"), (3, 4, "LOC")]
assert decode_spans(["O", "O"]) == []
assert canonical_bio(["I-PER", "I-LOC"]) == ["B-PER", "B-LOC"]

## 把原词与 BERT 子词对齐

对每个原词只在第一个子词上计算损失并读取预测。特殊符号、填充和后续子词的监督标签设为 `-100`。长句在整词之间切块，预测后按原句顺序拼回，保留跨块实体。极少数无法编码或单个词就超过窗口的输入以一个 `[UNK]` 表示，仍保留词的位置和标签。批次内动态补齐。[Hugging Face 词分类指南](https://huggingface.co/docs/transformers/tasks/token_classification)

In [ ]:
from functools import partial
from pathlib import Path
import hashlib
import json
import math
import platform
import tempfile
import time

import torch
import transformers
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer,
    get_linear_schedule_with_warmup, set_seed,
)


def encode_rows(tokenizer, rows, max_length):
    """每块记录句号、起始词索引和首子词位置，以便恢复完整句子。"""
    if not tokenizer.is_fast or not 3 <= max_length <= tokenizer.model_max_length:
        raise ValueError("请使用 fast BERT tokenizer，并检查 MAX_LENGTH。")
    capacity = max_length - 2  # BERT 单句的 [CLS] 和 [SEP]
    chunks = []
    for sentence, row in enumerate(rows):
        words = row["tokens"]
        if not words:
            continue
        encoding = tokenizer(
            words, is_split_into_words=True, add_special_tokens=False,
            truncation=False, return_attention_mask=False,
            return_token_type_ids=False, verbose=False,
        )
        pieces = [[] for _ in words]
        for token, index in zip(encoding["input_ids"], encoding.word_ids(), strict=True):
            pieces[index].append(token)

        start = 0
        while start < len(words):
            ids, targets, first = [tokenizer.cls_token_id], [-100], []
            end = start
            while end < len(words):
                group = pieces[end]
                if not group or len(group) > capacity:
                    group = [tokenizer.unk_token_id]
                if len(ids) + len(group) + 1 > max_length:
                    break
                first.append(len(ids))
                ids.extend(group)
                label = row["ner_tags"][end] if "ner_tags" in row else -100
                targets.extend([label] + [-100] * (len(group) - 1))
                end += 1
            ids.append(tokenizer.sep_token_id)
            targets.append(-100)
            chunks.append({"input_ids": ids, "labels": targets, "first": first,
                           "sentence": sentence, "start": start})
            start = end
    return chunks


def collate_chunks(chunks, pad_id):
    width = max(len(chunk["input_ids"]) for chunk in chunks)
    ids = torch.full((len(chunks), width), pad_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full_like(ids, -100)
    for i, chunk in enumerate(chunks):
        n = len(chunk["input_ids"])
        ids[i, :n] = torch.tensor(chunk["input_ids"])
        mask[i, :n] = 1
        labels[i, :n] = torch.tensor(chunk["labels"])
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}, chunks


@torch.inference_mode()
def predict_encoded(model, tokenizer, chunks, lengths):
    device = next(model.parameters()).device
    model.eval()
    sequences = [[] for _ in lengths]
    loader = DataLoader(chunks, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=partial(collate_chunks, pad_id=tokenizer.pad_token_id))
    for batch, records in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits
        ids = logits.argmax(-1).cpu().tolist()
        for row_ids, chunk in zip(ids, records, strict=True):
            seq = sequences[chunk["sentence"]]
            if len(seq) != chunk["start"]:
                raise RuntimeError("原词的顺序或位置发生变化。")
            seq.extend(model.config.id2label[row_ids[i]] for i in chunk["first"])
    if [len(seq) for seq in sequences] != lengths:
        raise RuntimeError("部分原词缺失预测。")
    return sequences


def predict_sequences(model, tokenizer, token_lists):
    """每个输入词对应一个原始 BIO 预测，按句返回。"""
    rows = [{"tokens": words} for words in token_lists]
    chunks = encode_rows(tokenizer, rows, model.config.ner_max_length)
    return predict_encoded(model, tokenizer, chunks, [len(row["tokens"]) for row in rows])

### 在训练前验证对齐

用一个只有八个子词位置的短窗口强制切块，并检查一个会被拆成多个子词的原词、标点及空句。这个检查只加载分词器。

In [ ]:
base_folder = Path(MODEL_ID)
base_options = {} if base_folder.is_dir() else {"revision": MODEL_REVISION}
local_base_sha256 = None
if base_folder.is_dir():
    # 离线模型可被同名替换；把权重、配置和分词器内容一起纳入运行签名。
    digest = hashlib.sha256()
    for asset in sorted(base_folder.iterdir()):
        if asset.is_file() and asset.suffix in {".safetensors", ".bin", ".json", ".txt", ".model"}:
            digest.update(asset.name.encode("utf-8") + b"\0")
            digest.update(asset.stat().st_size.to_bytes(8, "big"))
            with asset.open("rb") as handle:
                for block in iter(lambda: handle.read(4 * 1024**2), b""):
                    digest.update(block)
    local_base_sha256 = digest.hexdigest()

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, **base_options)
probe_tokens = ["electroencephalographically", ",", "Hong", "Kong", "University"] * 4
probe_gold = [i % len(LABELS) for i in range(len(probe_tokens))]
probe_chunks = encode_rows(base_tokenizer, [{"tokens": probe_tokens, "ner_tags": probe_gold}], 8)
assert len(probe_chunks) > 1 and all(len(c["input_ids"]) <= 8 for c in probe_chunks)
assert [c["labels"][i] for c in probe_chunks for i in c["first"]] == probe_gold
assert all(all(label == -100 for i, label in enumerate(c["labels"]) if i not in c["first"])
           for c in probe_chunks)
assert encode_rows(base_tokenizer, [{"tokens": []}], 8) == []
wide_chunks = encode_rows(base_tokenizer, [{"tokens": probe_tokens[:2], "ner_tags": probe_gold[:2]}], 32)
assert wide_chunks[0]["first"][1] - wide_chunks[0]["first"][0] > 1
print("通过：首子词标签、跨块还原、多子词和空句。")

## 分别微调 E2 和 E4

两个条件都从同一份预训练权重和固定种子开始，每轮结束用完整验证集的实体级 micro-F1 选模型，同分保留较早的一轮。CUDA 上开启 FP16 混合精度。训练期间可从进度条查看剩余时间。

每个已完成的条件保存到独立目录，包含最佳权重、分词器、训练日志和运行配置。重复执行会复用配置和数据完全匹配的结果。更改超参数时请选新的输出目录。若训练中断，重新运行该条件会从基础权重新开始，已有的临时日志和最佳权重保留在目录名含 `pending` 的文件夹中。

In [ ]:
run_dirs = {name: RUNS_DIR / name for name in ("E2", "E4")}


def dataset_digest(rows):
    digest = hashlib.sha256()
    for row in rows:
        entry = [row["id"], row["tokens"], row["ner_tags"]]
        digest.update(json.dumps(entry, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def train_condition(name, rows, provenance=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    settings = {
        "recipe": "conll-bert-first-subword-v1", "condition": name,
        "base_model": str(MODEL_ID), "base_revision": base_options.get("revision"),
        "base_local_assets_sha256": local_base_sha256,
        "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "max_length": MAX_LENGTH,
        "optimizer": "AdamW", "weight_decay": 0.01, "warmup_ratio": 0.1,
        "gradient_clip": 1.0, "checkpoint_metric": "validation entity micro-F1",
        "labels": list(LABELS), "train_sentences": len(rows),
        "train_sha256": dataset_digest(rows),
        "validation_sha256": dataset_digest(splits["validation"]),
        "torch": str(torch.__version__), "transformers": transformers.__version__,
        "mixed_precision": "fp16" if device.type == "cuda" else "off",
    }
    output = run_dirs[name]
    if output.exists():
        record = output / "training.json"
        if record.is_file():
            stored = json.loads(record.read_text(encoding="utf-8"))
            required = ("config.json", "model.safetensors", "tokenizer.json")
            if stored["settings"] == settings and all((output / "best" / f).is_file() for f in required):
                print(f"{name} 已完成，复用 {output}（第 {stored['best_epoch']} 轮）。")
                return
        raise RuntimeError(f"{output} 已存在且不匹配。请为这次运行选择新的 RUNS_DIR。")
    if not (rows and EPOCHS >= 1 and BATCH_SIZE >= 1):
        raise ValueError("请检查训练集、轮数和批量设置。")
    output.parent.mkdir(parents=True, exist_ok=True)
    pending = Path(tempfile.mkdtemp(dir=output.parent, prefix=f"{name}-pending-"))
    print(f"{name} 运行目录：{pending}")

    set_seed(SEED)
    torch.backends.cudnn.benchmark = False
    started = time.monotonic()
    tokenizer = base_tokenizer
    train_chunks = encode_rows(tokenizer, rows, MAX_LENGTH)
    validation = splits["validation"]
    val_chunks = encode_rows(tokenizer, validation, MAX_LENGTH)
    val_lengths = [len(row["tokens"]) for row in validation]
    train_loader = DataLoader(train_chunks, batch_size=BATCH_SIZE, shuffle=True,
                              generator=torch.Generator().manual_seed(SEED),
                              collate_fn=partial(collate_chunks, pad_id=tokenizer.pad_token_id))
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_ID, num_labels=len(LABELS), **base_options,
        label2id=label2id, id2label=id2label,
    ).to(device)
    model.config.ner_max_length = MAX_LENGTH
    if MAX_LENGTH > model.config.max_position_embeddings:
        raise ValueError("MAX_LENGTH 超过模型支持的序列长度。")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=settings["weight_decay"])
    steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, math.ceil(settings["warmup_ratio"] * steps), steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    history, best_score, best_epoch = [], -1.0, None
    for epoch in range(1, EPOCHS + 1):
        t0 = time.monotonic()
        model.train()
        total_loss, word_count = 0.0, 0
        for batch, records in tqdm(train_loader, desc=f"{name} · {epoch}/{EPOCHS}", leave=False):
            optimizer.zero_grad(set_to_none=True)
            inputs = {key: tensor.to(device) for key, tensor in batch.items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                loss = model(**inputs).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), settings["gradient_clip"])
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()
            count = sum(len(chunk["first"]) for chunk in records)
            total_loss += loss.detach().item() * count
            word_count += count
        predictions = predict_encoded(model, tokenizer, val_chunks, val_lengths)
        scores = score_predictions(validation, predictions)
        f1 = float(scores["micro_f1"])
        if not math.isfinite(f1) or not math.isfinite(total_loss):
            raise RuntimeError("训练损失或验证分数异常，请检查学习率与数据。")
        epoch_record = {"epoch": epoch, "train_loss": total_loss / word_count,
                        "validation_precision": float(scores["precision"]),
                        "validation_recall": float(scores["recall"]),
                        "validation_micro_f1": f1, "seconds": round(time.monotonic() - t0, 2)}
        history.append(epoch_record)
        with (pending / "epochs.jsonl").open("a", encoding="utf-8") as log:
            print(json.dumps(epoch_record), file=log)
        print(f"{name} · 第 {epoch} 轮 · 验证 micro-F1 {f1:.4f} · {epoch_record['seconds']} 秒")
        if f1 > best_score:
            best_score, best_epoch = f1, epoch
            model.save_pretrained(pending / "best", safe_serialization=True)
            tokenizer.save_pretrained(pending / "best")

    record = {
        "settings": settings, "subset": provenance, "epochs": history, "best_epoch": best_epoch,
        "best_validation_micro_f1": best_score, "seconds": round(time.monotonic() - started, 2),
        "python": platform.python_version(), "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU",
        "peak_allocated_mib": round(torch.cuda.max_memory_allocated(device) / 2**20)
                              if device.type == "cuda" else None,
        "base_commit_loaded": getattr(model.config, "_commit_hash", None),
        "train_chunks": len(train_chunks), "validation_chunks": len(val_chunks),
    }
    (pending / "training.json").write_text(json.dumps(record, ensure_ascii=False, indent=2, default=str),
                                            encoding="utf-8")
    pending.rename(output)
    print(f"{name} 已保存到 {output}；最佳轮次 {best_epoch}。")

### E2 · 全量 BERT

拿到原始数据后就能单独执行这个条件。

In [ ]:
train_condition("E2", splits["train"])

### E4 · 固定 10% BERT

把 A 确认的 ID 清单放在数据配置指定的位置，再单独运行下面一格。E4 从原始 BERT 权重独立开始训练。

In [ ]:
train_10pct, subset_provenance = load_shared_subset()
if train_10pct is None:
    print("E4 待运行：请先上传 A 确认的 10% 训练句 ID 清单，再重新运行本格。")
else:
    train_condition("E4", train_10pct, subset_provenance)

## 测试预测与成绩交接

本单元从磁盘重新加载每个已完成实验的最佳模型，生成完整测试集预测。每行 JSON 含 `id`、`tokens`、`pred_tags`，后者保留模型原始输出，供 A 在共用解码器里修正无效 BIO。这里按小组协议另算一份复核分数，并使用 seqeval 严格 IOB2 模式交叉核对。模型选择均已在前面的验证集步骤完成。

In [ ]:
import csv
import gc
import json
import math
from pathlib import Path
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score, precision_score, recall_score
from seqeval.scheme import IOB2

def write_json(path, payload):
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, records):
    with path.open("w", newline="", encoding="utf-8") as out:
        writer = csv.DictWriter(out, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

def load_best(run):
    folder = run_dirs[run] / "best"
    tokenizer = AutoTokenizer.from_pretrained(folder, use_fast=True, local_files_only=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForTokenClassification.from_pretrained(folder, local_files_only=True).to(device).eval()
    return model, tokenizer

def export_predictions(run, rows):
    folder = run_dirs[run]
    model, tokenizer = load_best(run)
    try:
        outputs = predict_sequences(model, tokenizer, [r["tokens"] for r in rows])
    finally:
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    scores = score_predictions(rows, outputs)
    truth = [canonical_bio([LABELS[i] for i in r["ner_tags"]]) for r in rows]
    decoded = [canonical_bio(seq) for seq in outputs]
    for key, metric in (("precision", precision_score), ("recall", recall_score), ("micro_f1", f1_score)):
        crosscheck = metric(truth, decoded, mode="strict", scheme=IOB2, zero_division=0)
        if not math.isclose(float(crosscheck), scores[key], abs_tol=1e-10):
            raise ValueError(f"{run} 的 {key} 与 seqeval 结果不符")
    out_path = folder / "test_predictions.jsonl"
    with out_path.open("w", encoding="utf-8") as out:
        for row, tags in zip(rows, outputs):
            record = {"id": str(row["id"]), "tokens": row["tokens"], "pred_tags": tags}
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
    write_json(folder / "test_scores.json", scores)
    return scores

available = ["E2"] + (["E4"] if train_10pct is not None else [])
results = {run: export_predictions(run, splits["test"]) for run in available}
summary = [{"experiment": run, "train_sentences": len(splits["train"]) if run == "E2" else len(train_10pct),
            **{key: values[key] for key in ("precision", "recall", "micro_f1", "tp", "fp", "fn")}}
           for run, values in results.items()]
for row in summary:
    print(f"{row['experiment']}   P={row['precision']:.4f}   R={row['recall']:.4f}   F1={row['micro_f1']:.4f}")
if "E4" in results:
    print(f"BERT 数据缩减后的变化 E2 − E4 = {100 * (results['E2']['micro_f1'] - results['E4']['micro_f1']):+.2f} F1 点")

## 给 A 的接口，给 B 的演示

`predict_tags(tokens)` 接受 A 已分好的原词列表，每个词返回一个标签。下面的 `predict(text)` 为英文演示提供分词和字符位置，用开始词和结束词的字符偏移从原句切出原文。小组集成时直接复用 A 的原文分词与评分。演示的期望覆盖组织、多个实体和无实体三种情形，输出以实际模型为准。

In [ ]:
import re

DEMO_RUN = "E2"
demo_model, demo_tokenizer = load_best(DEMO_RUN)
# 保留缩写、词内撇号和连字符，单独切出其余标点；保存每个原词的字符区间。
word_pattern = re.compile(r"(?:[A-Za-z]\.){2,}|\w+(?:['’\-]\w+)*|[^\w\s]", flags=re.UNICODE)

def predict_tags(tokens):
    if not tokens:
        return []
    return predict_sequences(demo_model, demo_tokenizer, [list(tokens)])[0]

def predict(text):
    words = list(word_pattern.finditer(text))
    tags = predict_tags([w.group() for w in words])
    return [{"text": text[words[start].start():words[end - 1].end()], "type": kind}
            for start, end, kind in decode_spans(tags)]

for sentence in ("I study at City University of Hong Kong.",
                 "Alice met David in London.", "It is raining today."):
    print(sentence)
    print(json.dumps(predict(sentence), ensure_ascii=False))

## 下载交接材料

**轻量交接包**给 A 和 B，含每组测试预测、指标 CSV、按长度和类别的计数、训练日志与配置，以及 E4 实际使用的清单。**模型包**按实验单独保存，含最佳模型和分词器。把模型包解压至对应实验的 `best/` 即可用上面的预测接口演示。训练文件夹及笔记本也请保留，便于复现。

In [ ]:
import zipfile

delivery = WORK_DIR / "delivery"
delivery.mkdir(parents=True, exist_ok=True)
handoff_files = {}
for run in available:
    folder = run_dirs[run]
    for name in ("test_predictions.jsonl", "test_scores.json", "training.json", "epochs.jsonl"):
        handoff_files[f"{run}/{name}"] = folder / name
    weights = delivery / f"{run}_best_model.zip"
    pending = weights.with_suffix(".zip.partial")
    with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted((folder / "best").rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(folder / "best").as_posix())
    pending.replace(weights)
    print(f"{run} 模型包 · {weights} · {weights.stat().st_size / 1024**2:.1f} MiB")

import tempfile
with tempfile.TemporaryDirectory(dir=delivery) as scratch:
    tmp = Path(scratch)
    write_csv(tmp / "metrics.csv", summary)
    type_table = [{"experiment": r, "type": t, **m} for r, s in results.items() for t, m in s["per_type"].items()]
    length_table = [{"experiment": r, "words": t, **m} for r, s in results.items() for t, m in s["by_length"].items()]
    write_csv(tmp / "metrics_by_type.csv", type_table)
    write_csv(tmp / "recall_by_length.csv", length_table)
    write_json(tmp / "manifest.json", {
        "experiments": available, "dataset_revision": DATA_REVISION, "data": data_provenance,
        "model": str(MODEL_ID), "model_revision": base_options.get("revision"),
        "local_base_assets_sha256": local_base_sha256,
        "e4_subset": subset_provenance, "labels_in_id_order": LABELS,
        "counts": {k: len(v) for k, v in splits.items()},
        "metric_scale": "0..1", "invalid_I_policy": "start a new entity", "span_end": "exclusive",
    })
    if "E4" in available:
        write_json(tmp / "train_10pct_ids.json", [str(r["id"]) for r in train_10pct])
    for path in tmp.iterdir():
        handoff_files[path.name] = path
    archive_name = delivery / ("BERT_C_" + "_".join(available) + "_handoff.zip")
    pending = archive_name.with_suffix(".zip.partial")
    with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=5) as archive:
        for name, path in handoff_files.items():
            archive.write(path, name)
    pending.replace(archive_name)
print(f"给 A 和 B 的交接包 · {archive_name}")
print("用左侧文件面板下载以上 zip。需要长期保存时，也可复制到课程授权的 Cloud Storage 存储桶。")

### 用 Cloud Storage 保存（可选）

在运行时的终端中可运行 `gcloud storage cp /实际路径/包名.zip gs://你的课程存储桶/你的目录/`。将命令中的源路径替换为上面打印的单个文件路径。使用老师已授权的存储桶。结束作业后在 Colab Enterprise 的 **Runtimes** 页面停止自己使用的运行时。